# Logistic regression — goi tu contest_kit

Logic nam trong `contest_kit/logreg.py`, notebook chi goi ham.
Ban viet tay ban dau con trong git: `git show 667f6f2:notebooks/logistic_regression.ipynb`.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from contest_kit.logreg import fit, predict, predict_proba, bce_loss

RAW = pd.read_csv("../data/digit-recognizer/train.csv")
print(RAW.shape)

## Lay mot cap chu so

In [ ]:
def lay_cap(a, b, n_tr=5000, n_val=1000, seed=0):
    """Loc con hai chu so a, b. Nhan: a -> 0, b -> 1. Chia 255 ngay tai day."""
    t = RAW[RAW["label"].isin([a, b])]
    y = (t["label"] == b).astype(int)
    X = t.drop(columns="label") / 255.0
    return train_test_split(X, y, train_size=n_tr, test_size=n_val,
                            stratify=y, random_state=seed)


X_tr, X_val, y_tr, y_val = lay_cap(0, 1)
X_tr.shape, y_tr.mean()

## Huan luyen

In [ ]:
W, b, hist = fit(X_tr, y_tr, lr=0.1, epoch=100, X_val=X_val, y_val=y_val)

acc = (predict(X_val, W, b) == np.asarray(y_val)).mean()
print(f"val accuracy : {acc:.4f}")
print(f"val loss     : {bce_loss(predict_proba(X_val, W, b), y_val):.6f}")
print(f"loss epoch 0 : {hist['train'][0]:.6f}   (= ln 2 vi W = 0)")
print(f"loss cuoi    : {hist['train'][-2]:.6f} -> {hist['train'][-1]:.6f}")

## Da hoi tu chua?

Hai duong con di xuong o cuoi nghia la **thieu epoch**, khong phai het cach.
Duong val quay dau di len moi la overfit.

In [ ]:
plt.plot(hist["train"], label="train")
plt.plot(hist["val"], label="val")
plt.xlabel("epoch"); plt.ylabel("BCE loss"); plt.legend(); plt.show()

## Cap de che mat moi thu

`0 vs 1` de den muc moi cach lam deu trong giong nhau. Doi cap moi thay khoang cach.

In [ ]:
for a, b_ in [(0, 1), (3, 5), (4, 9)]:
    Xa, Xb, ya, yb = lay_cap(a, b_)
    W_, b2, _ = fit(Xa, ya, lr=0.1, epoch=100)
    print(f"{a} vs {b_}: {(predict(Xb, W_, b2) == np.asarray(yb)).mean():.4f}")